# Predict Customer Churn
## Score: .91357

In [29]:
N_FOLDS = 5
N_SEEDS = 3
OPTUNA_TRIALS = 50

In [30]:
import numpy as np
import pandas as pd

train = pd.read_csv("playground-series-s6e3/train.csv")
test = pd.read_csv("playground-series-s6e3/test.csv")
original = pd.read_csv("original_data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
original = original.drop(columns=['customerID'])
original = original[train.columns.drop('id')]
train_combined = pd.concat([train.drop(columns=['id']), original], ignore_index=True)
sample_weights = np.array([1.0] * len(train) + [0.5] * len(original))
y_full = train_combined['Churn'].map({'Yes': 1, 'No': 0})
X_full = train_combined.drop(columns=['Churn'])
test_ids = test['id']
X_test_raw = test.drop(columns=['id'])
print(f"Train: {len(train)}, Original: {len(original)}, Combined: {len(X_full)}")

Train: 594194, Original: 7043, Combined: 601237


In [31]:
def engineer_features(df, total_charges_median=None):
    df = df.copy()
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    median_tc = total_charges_median if total_charges_median is not None else df['TotalCharges'].median()
    df['TotalCharges'] = df['TotalCharges'].fillna(median_tc)
    df['AvgChargesPerMonth'] = df['TotalCharges'] / df['tenure'].replace(0, 1)
    df['MonthlyChargesSqrt'] = np.sqrt(df['MonthlyCharges'])
    df['tenure_squared'] = df['tenure'] ** 2
    df['ChargesRatio'] = df['TotalCharges'] / (df['MonthlyCharges'] * (df['tenure'] + 1))
    df['Contract_MonthToMonth'] = (df['Contract'] == 'Month-to-month').astype(int)
    df['Contract_OneYear'] = (df['Contract'] == 'One year').astype(int)
    df['Contract_TwoYear'] = (df['Contract'] == 'Two year').astype(int)
    df['IsFiberOptic'] = (df['InternetService'] == 'Fiber optic').astype(int)
    df['NumStreaming'] = ((df['StreamingTV'] == 'Yes') | (df['StreamingMovies'] == 'Yes')).astype(int)
    df['NumStreamingBoth'] = ((df['StreamingTV'] == 'Yes') & (df['StreamingMovies'] == 'Yes')).astype(int)
    addon_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']
    df['NumAddons'] = sum((df[c] == 'Yes').astype(int) for c in addon_cols)
    df['PaymentElectronic'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['HasDependents'] = (df['Dependents'] == 'Yes').astype(int)
    df['HasPartner'] = (df['Partner'] == 'Yes').astype(int)
    df['SeniorWithShortTenure'] = (df['SeniorCitizen'] == 1) & (df['tenure'] < 12)
    df['SeniorWithShortTenure'] = df['SeniorWithShortTenure'].astype(int)
    df['HighChargeShortTenure'] = (df['MonthlyCharges'] > 70) & (df['tenure'] < 12)
    df['HighChargeShortTenure'] = df['HighChargeShortTenure'].astype(int)
    return df

def target_encode(train_df, test_df, col, target_series, m=5):
    global_mean = target_series.mean()
    combined = pd.DataFrame({col: train_df[col], '_y': target_series.values})
    agg = combined.groupby(col)['_y'].agg(['mean', 'count'])
    smooth = (agg['mean'] * agg['count'] + global_mean * m) / (agg['count'] + m)
    train_df = train_df.copy()
    test_df = test_df.copy()
    train_df[f'{col}_te'] = train_df[col].map(smooth).fillna(global_mean)
    test_df[f'{col}_te'] = test_df[col].map(smooth).fillna(global_mean)
    return train_df, test_df

X_full = engineer_features(X_full)
tc_median = X_full['TotalCharges'].median()
X_test_raw = engineer_features(X_test_raw, total_charges_median=tc_median)

for col in ['Contract', 'PaymentMethod', 'InternetService']:
    X_full, X_test_raw = target_encode(X_full, X_test_raw, col, y_full, m=20)
    X_full = X_full.drop(columns=[col])
    X_test_raw = X_test_raw.drop(columns=[col])

X_encoded = pd.get_dummies(X_full, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_raw, drop_first=True)
X_test_encoded = X_test_encoded.reindex(columns=X_encoded.columns, fill_value=0)

In [32]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fit_kw = {'sample_weight': sample_weights}

In [33]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 700),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = XGBClassifier(**params, random_state=42, eval_metric='logloss')
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_xgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_xgb.optimize(xgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_xgb_params = study_xgb.best_params
print(f"XGB Best CV AUC: {study_xgb.best_value:.4f}")

[I 2026-03-03 23:01:31,188] A new study created in memory with name: no-name-f0300294-9888-4c37-a4b9-81748607af16


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-03 23:02:42,910] Trial 0 finished with value: 0.9145277282981146 and parameters: {'n_estimators': 625, 'max_depth': 4, 'learning_rate': 0.023399763820895154, 'subsample': 0.817492869081369, 'colsample_bytree': 0.9220361730679456, 'scale_pos_weight': 1.5806424570858142, 'reg_alpha': 0.006286895636028893, 'reg_lambda': 7.805896525630794, 'min_child_weight': 2}. Best is trial 0 with value: 0.9145277282981146.
[I 2026-03-03 23:03:34,267] Trial 1 finished with value: 0.9147240520203294 and parameters: {'n_estimators': 294, 'max_depth': 8, 'learning_rate': 0.09467776687692245, 'subsample': 0.7145465456542065, 'colsample_bytree': 0.7057677431826627, 'scale_pos_weight': 1.2425250862866153, 'reg_alpha': 0.3662418378294923, 'reg_lambda': 0.0015045646362584477, 'min_child_weight': 3}. Best is trial 1 with value: 0.9147240520203294.
[I 2026-03-03 23:04:14,464] Trial 2 finished with value: 0.9152525500878663 and parameters: {'n_estimators': 286, 'max_depth': 6, 'learning_rate': 0.1397759

In [34]:
from lightgbm import LGBMClassifier

def lgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 700),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = LGBMClassifier(**params, random_state=42, verbose=-1)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_lgb = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_lgb.optimize(lgb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_lgb_params = study_lgb.best_params
print(f"LGB Best CV AUC: {study_lgb.best_value:.4f}")

[I 2026-03-03 23:50:41,878] A new study created in memory with name: no-name-81e4baa0-dade-48d9-a46c-a0db28b504bb


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-03 23:51:48,268] Trial 0 finished with value: 0.9152552472525214 and parameters: {'n_estimators': 534, 'max_depth': 6, 'learning_rate': 0.048631390970802046, 'subsample': 0.668566700673678, 'colsample_bytree': 0.930968294996324, 'scale_pos_weight': 3.598066709749102, 'reg_alpha': 0.028108115924800288, 'reg_lambda': 0.036821608620889734, 'min_child_samples': 6}. Best is trial 0 with value: 0.9152552472525214.
[I 2026-03-03 23:52:28,160] Trial 1 finished with value: 0.9153849045434944 and parameters: {'n_estimators': 693, 'max_depth': 3, 'learning_rate': 0.08571564379543371, 'subsample': 0.6940583249527483, 'colsample_bytree': 0.7851855658104124, 'scale_pos_weight': 2.297252586407744, 'reg_alpha': 1.7597721409114349, 'reg_lambda': 6.737630932219996, 'min_child_samples': 72}. Best is trial 1 with value: 0.9153849045434944.
[I 2026-03-03 23:52:57,669] Trial 2 finished with value: 0.9145726985682682 and parameters: {'n_estimators': 314, 'max_depth': 4, 'learning_rate': 0.05071244

In [35]:
from catboost import CatBoostClassifier

def cat_objective(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 700),
        'depth': trial.suggest_int('depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.15),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.0, 4.0),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
    }
    scores = []
    for train_idx, val_idx in cv.split(X_encoded, y_full):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr, y_val = y_full.iloc[train_idx], y_full.iloc[val_idx]
        sw_tr = sample_weights[train_idx]
        m = CatBoostClassifier(**params, random_state=42, verbose=0)
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        scores.append(roc_auc_score(y_val, m.predict_proba(X_val)[:, 1]))
    return np.mean(scores)

study_cat = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(n_startup_trials=10))
study_cat.optimize(cat_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_cat_params = study_cat.best_params
print(f"CatBoost Best CV AUC: {study_cat.best_value:.4f}")

[I 2026-03-04 00:29:58,859] A new study created in memory with name: no-name-b81fdd85-6811-4c64-8859-7878bff87668


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-04 00:32:47,878] Trial 0 finished with value: 0.913933247814947 and parameters: {'iterations': 620, 'depth': 8, 'learning_rate': 0.08172466600579653, 'subsample': 0.6571546869461268, 'colsample_bylevel': 0.855731407380573, 'scale_pos_weight': 3.684772238629215, 'l2_leaf_reg': 0.0013214158519579732}. Best is trial 0 with value: 0.913933247814947.
[I 2026-03-04 00:33:31,227] Trial 1 finished with value: 0.9144554984166042 and parameters: {'iterations': 238, 'depth': 4, 'learning_rate': 0.10645387514904309, 'subsample': 0.6603369420949505, 'colsample_bylevel': 0.8146181860834619, 'scale_pos_weight': 2.6005254234056703, 'l2_leaf_reg': 0.5548556637932266}. Best is trial 1 with value: 0.9144554984166042.
[I 2026-03-04 00:34:47,474] Trial 2 finished with value: 0.91541884906606 and parameters: {'iterations': 309, 'depth': 6, 'learning_rate': 0.13732782618612108, 'subsample': 0.9468162359577443, 'colsample_bylevel': 0.8727502484371982, 'scale_pos_weight': 2.20740334140231, 'l2_leaf_

In [36]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

n_models = 5
oof = np.zeros((len(X_encoded), n_models))
test_preds = np.zeros((len(X_test_encoded), n_models))

def get_models(seed, fold):
    return [
        XGBClassifier(**best_xgb_params, random_state=seed+fold, eval_metric='logloss'),
        LGBMClassifier(**best_lgb_params, random_state=seed+fold, verbose=-1),
        CatBoostClassifier(**best_cat_params, random_state=seed+fold, verbose=0),
        RandomForestClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
        ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=seed+fold),
    ]

for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
    X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
    y_tr = y_full.iloc[train_idx]
    sw_tr = sample_weights[train_idx]
    models = get_models(42, fold)
    for i, m in enumerate(models):
        m.fit(X_tr, y_tr, sample_weight=sw_tr)
        oof[val_idx, i] = m.predict_proba(X_val)[:, 1]
        test_preds[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS

def neg_auc(w):
    blend = oof @ w
    return -roc_auc_score(y_full, blend)

result = minimize(neg_auc, x0=np.ones(n_models)/n_models, method='SLSQP',
                  bounds=[(0, 1)]*n_models,
                  constraints={'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
blend_weights = result.x
blend_oof = oof @ blend_weights
print(f"Blend OOF AUC: {roc_auc_score(y_full, blend_oof):.4f}")
print(f"Weights: {dict(zip(['XGB','LGB','Cat','RF','ET'], np.round(blend_weights, 4)))}")

Blend OOF AUC: 0.9155
Weights: {'XGB': np.float64(0.2116), 'LGB': np.float64(0.2112), 'Cat': np.float64(0.2291), 'RF': np.float64(0.1629), 'ET': np.float64(0.1851)}


In [37]:
all_test_preds = [test_preds @ blend_weights]
for base_seed in [123, 456, 789][:N_SEEDS-1]:
    tp = np.zeros((len(X_test_encoded), n_models))
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_encoded, y_full)):
        X_tr, X_val = X_encoded.iloc[train_idx], X_encoded.iloc[val_idx]
        y_tr = y_full.iloc[train_idx]
        sw_tr = sample_weights[train_idx]
        models = get_models(base_seed, fold)
        for i, m in enumerate(models):
            m.fit(X_tr, y_tr, sample_weight=sw_tr)
            tp[:, i] += m.predict_proba(X_test_encoded)[:, 1] / N_FOLDS
    all_test_preds.append(tp @ blend_weights)

final_preds = np.mean(all_test_preds, axis=0)
submission = pd.DataFrame({'id': test_ids, 'Churn': final_preds})
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Churn
0,594194,0.125399
1,594195,0.001592
2,594196,0.198022
3,594197,0.006960
4,594198,0.652834
